In [ ]:
import polars as pl
from settings import load_settings

In [ ]:
SETTINGS = load_settings()

In [ ]:
init_df = pl.scan_parquet(SETTINGS.initial_dataset_path)

In [ ]:
init_df.collect_schema()

In [ ]:
init_df.head().collect()

In [ ]:
#init_df.select(pl.col('package_name')).unique().count().collect()

In [ ]:
#init_df.select(pl.col('package_name'), pl.col('github_repo')).unique().count().collect()

In [ ]:
df = (
    init_df
    .select(
        pl.col('github_repo'), 
        pl.col('package_name'), 
        pl.col("package_version"), 
        pl.col("dep_name")
    )
    .group_by(pl.col('github_repo'), pl.col('package_name'), pl.col("package_version")).agg(pl.col("dep_name").n_unique().alias("num_deps"))
    #.head()
    #.collect()
)

#df

In [ ]:
df.explain()

In [ ]:
df.sink_parquet("/workspaces/mads-siads-699-winter-2026-capstone/notebooks/data/augmented_data/feature_num_deps.parquet")

In [ ]:
num_deps_df = pl.scan_parquet("/workspaces/mads-siads-699-winter-2026-capstone/notebooks/data/augmented_data/feature_num_deps.parquet")

In [ ]:
num_deps_df.head().collect()

In [ ]:
adopter_edges = (
    init_df
    .filter(pl.col("MinimumDepth") == 1) # only direct dependencies
    .select(["package_name", "package_version", "dep_name", "dep_version"])
    .drop_nulls(["package_name", "package_version", "dep_name", "dep_version"])
    .unique() # removes duplicates across snapshots
)


In [ ]:
adopter_edges.collect().head(100)

In [ ]:
adopter_edges.schema

In [ ]:
popularity_no_version = (
    adopter_edges
    .select([
        "package_name",
        "dep_name",
    ])
    .group_by("dep_name")
    .agg(
        adopting_packages=pl.n_unique("package_name")
    )
    .sort("adopting_packages", descending=True)
    .rename({
        "dep_name": "package_name",
        "adopting_packages": "dependency_count"
    })
    .collect()
)

In [ ]:
popularity_no_version.head(100)

In [ ]:
popularity_with_adopter_versions = (
    adopter_edges
    .group_by(["dep_name", "dep_version"])
    .agg(
        adopting_package_versions=pl.len()
    )
    .sort("adopting_package_versions", descending=True)
    .rename({
        "dep_name": "package_name",
        "dep_version": "package_version",
        "adopting_package_versions": "dependency_count"
    })
    .collect()
)

In [ ]:
popularity_with_adopter_versions.schema

In [ ]:
popularity_with_adopter_versions.head(100)